### In this notebook, we train two xgboost models to predict whether or not a customer will default in future.
- The first xgboost model is trained with given numerical features as is.
- The second xgboost model is trained with given numerical features as well as the Autoregressive RNN generated features.
- Save the model of the 2nd xgboost and write configure file for triton inference

In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [2]:
import cudf
import cupy
from tqdm import tqdm
import numpy as np
import gc
import xgboost as xgb
from utils import amex_metric_np

cudf.__version__, xgb.__version__

('25.10.00', '3.0.5')

In [3]:
PATH = 'data/amex'

# Data preprocessing

In [4]:
%%time
train = cudf.read_parquet(f'{PATH}/train.parquet')
trainl = cudf.read_csv(f'{PATH}/train_labels.csv')
print(trainl.shape)
trainl.head()

(458913, 2)
CPU times: user 461 ms, sys: 758 ms, total: 1.22 s
Wall time: 867 ms


,customer_ID,target
0,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,0
1,00000fd6641609c6ece5454664794f0340ad84dddce9a2...,0
2,00001b22f846c82c51f6e3958ccd81970162bae8b007e8...,0
3,000041bdba6ecadd89a52d11886e8eaaec9325906c9723...,0
4,00007889e4fcd2614b6cbe7f8f3d2e5c728eca32d9eb8a...,0


In [5]:
trainl['target'].value_counts()

target
0    340085
1    118828
Name: count, dtype: int64

In [6]:
%%time
train = train.merge(trainl, on='customer_ID', how='left')
print(train.shape)
train.head()

(5531451, 191)
CPU times: user 28 ms, sys: 19 ms, total: 47 ms
Wall time: 46 ms


,customer_ID,S_2,P_2,D_39,B_1,B_2,R_1,S_3,D_41,B_3,...,D_137,D_138,D_139,D_140,D_141,D_142,D_143,D_144,D_145,target
0,03b5352be1d13ee336fe78b3ff563e3bdcdf344015a328...,2017-08-15,0.244338,26,0.059066,0.002359,0.007854,0.067195,0.255404,0.004431,...,-1,-1,0,0,0.0,<NA>,0,0.007661,0,0
1,03b5352be1d13ee336fe78b3ff563e3bdcdf344015a328...,2017-09-27,0.271979,38,0.066835,0.341636,0.002493,0.072573,0.559332,0.057856,...,-1,-1,0,0,0.0,<NA>,0,0.007022,0,0
2,03b5352be1d13ee336fe78b3ff563e3bdcdf344015a328...,2017-10-05,0.270212,46,0.050073,0.341233,0.500572,0.070022,0.557480,0.055727,...,-1,-1,0,0,0.0,<NA>,0,0.000127,0,0
3,03b5352be1d13ee336fe78b3ff563e3bdcdf344015a328...,2017-11-23,0.275013,4,0.039562,0.004197,0.009004,0.080363,0.587373,0.032080,...,-1,-1,0,0,0.0,<NA>,0,0.009850,0,0
4,03b5352be1d13ee336fe78b3ff563e3bdcdf344015a328...,2017-12-05,0.223527,16,0.052601,0.002427,0.503076,0.083620,0.586695,0.029603,...,-1,-1,0,0,0.0,<NA>,0,0.009735,0,0


In [7]:
%%time

train['cid'], _ = train.customer_ID.factorize()
train['S_2'] = cudf.to_datetime(train['S_2'])

CPU times: user 26.1 ms, sys: 21.6 ms, total: 47.7 ms
Wall time: 47.1 ms


In [8]:
%%time

rnn_feas = np.load('rnn_feas.npy')
rnn_feas.shape

CPU times: user 1.6 ms, sys: 638 ms, total: 639 ms
Wall time: 638 ms


(458913, 13, 177)

In [9]:
mask = np.arange(rnn_feas.shape[0])%4==0

In [10]:
tr_rnn = rnn_feas[~mask]
va_rnn = rnn_feas[mask]
tr_rnn.shape, va_rnn.shape

((344184, 13, 177), (114729, 13, 177))

In [11]:
mask = train['cid']%4 == 0
tr,va = train.loc[~mask],train.loc[mask]
print("Verify target distribution is consistent across tr and va")
print(tr['target'].mean(), va['target'].mean())

Verify target distribution is consistent across tr and va
0.24858778579033497 0.2506255315210034


### Utility Functions

In [12]:
def get_cat_cols():
    return ['B_30', 'B_38', 'D_114', 'D_116', 'D_117', 'D_120',
                'D_126', 'D_63', 'D_64', 'D_66', 'D_68']

def preprocess(df):
    df = df.sort_values(['cid','S_2'])
    df = df.drop_duplicates('cid',keep='last')
    df = df.sort_values('cid')
    df = df.reset_index(drop=True)
    return df

In [13]:
%%time

tr = preprocess(tr)
print(tr.shape)
tr.head()

(344184, 192)
CPU times: user 25.7 ms, sys: 32.4 ms, total: 58.1 ms
Wall time: 57.8 ms


,customer_ID,S_2,P_2,D_39,B_1,B_2,R_1,S_3,D_41,B_3,...,D_138,D_139,D_140,D_141,D_142,D_143,D_144,D_145,target,cid
0,03b54326acef626a1004648d63544a5797156553c80159...,2018-03-03,0.702649,0,0.009530,1.001512,0.008194,0.084654,0.000000,0.010145,...,-1,1,0,0.890045,0.163437784,1,0.009677,2,0,1
1,03b5d09972f7d7564481ed6365a6d945f2e0c0242290a9...,2018-03-28,0.830973,9,0.027950,1.007413,0.008819,0.069935,0.000000,0.001502,...,-1,1,0,0.979590,0.557219565,1,0.263047,1,0,2
2,03b5f17e4bdb451b4ab1c7ca7495025ecc4e6e4f5746be...,2018-03-21,1.002720,5,0.005989,1.002686,0.001954,0.135628,0.000000,0.012428,...,-1,0,0,0.000000,<NA>,0,0.003631,0,0,3
3,0004860c260168fcaad0734a1dfedb7ceb1a83aaac54e2...,2018-03-09,0.074849,88,0.107940,0.054011,0.500593,1.109986,0.919821,0.180245,...,-1,1,0,0.875536,0.493616223,1,0.009901,12,1,5
4,009a9aa2dd8ac983d271107b71193ad1b99398ae65b488...,2018-03-28,0.259206,1,0.169090,0.160282,0.001875,0.173345,0.589237,0.132350,...,-1,0,0,0.000000,<NA>,0,0.000823,0,1,6


In [14]:
%%time

va = preprocess(va)
print(va.shape)
va.head()

(114729, 192)
CPU times: user 25.9 ms, sys: 11.4 ms, total: 37.3 ms
Wall time: 36.8 ms


,customer_ID,S_2,P_2,D_39,B_1,B_2,R_1,S_3,D_41,B_3,...,D_138,D_139,D_140,D_141,D_142,D_143,D_144,D_145,target,cid
0,03b5352be1d13ee336fe78b3ff563e3bdcdf344015a328...,2018-03-19,0.234197,0,0.023874,1.005708,0.007264,0.067244,0.734020,0.019120,...,-1,0,0,0.000000,<NA>,0,0.000153,0,0,0
1,0004837f0c785928a29a6f83f70f4a1c54caec483a773f...,2018-03-02,0.642295,23,0.429796,0.024450,0.005188,0.221216,0.000000,0.357104,...,-1,0,0,0.000000,<NA>,0,0.005832,0,0,4
2,009ac0e83fed952c73b13ba9f7b5de96428a0cfead74d8...,2018-03-12,0.797550,0,0.008773,1.006058,0.001309,0.211654,0.000000,0.010312,...,-1,0,0,0.000000,<NA>,0,0.002519,0,0,8
3,022c4192a191007ed75c4684aaca25cf2749fe292a5211...,2018-03-06,-0.044901,14,0.103454,0.053541,2.006770,0.507249,0.159375,0.209965,...,-1,1,1,0.892652,0.20759286,1,0.008953,4,1,12
4,0004ec03ca1ab2adb9aa260c61ba5dce8185e19d3ab704...,2018-03-08,0.980221,0,0.028153,1.008633,0.503520,0.130219,0.000000,0.006706,...,-1,0,0,0.000000,<NA>,0,0.007432,0,0,16


In [15]:
not_used = [i for i in tr.columns if i in ['cid','target','S_2'] or tr[i].dtype=='O']
not_used += get_cat_cols()

X_train = tr.drop(not_used,axis=1)
y_train = tr['target']

X_test = va.drop(not_used,axis=1)
y_test = va['target']

for i in X_train.columns:
    X_train[i] = X_train[i].astype('float32')
    X_test[i] = X_test[i].astype('float32')

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((344184, 177), (344184,), (114729, 177), (114729,))

In [16]:
X_train.columns

Index(['P_2', 'D_39', 'B_1', 'B_2', 'R_1', 'S_3', 'D_41', 'B_3', 'D_42',
       'D_43',
       ...
       'D_136', 'D_137', 'D_138', 'D_139', 'D_140', 'D_141', 'D_142', 'D_143',
       'D_144', 'D_145'],
      dtype='object', length=177)

In [17]:
del train,tr,va
gc.collect()

0

# Train the 1st xgboost model with given features only

In [18]:
def get_xgb_model():
    max_depth = 7
    num_trees = 1000
    early_stop = xgb.callback.EarlyStopping(rounds=10,
                                            maximize=True,
                                            metric_name='amex_metric_np',
                                            data_name='validation_0')
    model = xgb.XGBClassifier(
            tree_method='hist',
            device='cuda',
            enable_categorical=False,  
            #eval_metric='auc',
            objective='binary:logistic',
            max_depth=max_depth,
            n_estimators=num_trees,
            #colsample_bytree=0.5,
            min_child_weight=50,
            eval_metric=amex_metric_np,
            callbacks=[early_stop]
            #gamma=10,
    )
    return model

In [19]:
model = get_xgb_model()
model.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)],
        verbose=100
    )
model.best_score

[0]	validation_0-logloss:0.43122	validation_0-amex_metric_np:0.70869
[59]	validation_0-logloss:0.22728	validation_0-amex_metric_np:0.77586


0.776331

### The evaluation metric for this competition is the mean of two measures of rank ordering: Normalized Gini Coefficient, and default rate captured at 4%. The larger the metric, the more accurate the model is to predict the default in future. Please find the [description](https://www.kaggle.com/competitions/amex-default-prediction/overview/evaluation) and [analysis](https://www.kaggle.com/competitions/amex-default-prediction/overview/evaluation) to understand more about this metric. For now all we care is **larger metric is better!**

In [20]:
del model
gc.collect()

1168

# Add RNN features and train the xgboost again

In [21]:
%%time
rnn_feas = np.load('rnn_feas.npy')
rnn_feas.shape

CPU times: user 294 μs, sys: 634 ms, total: 635 ms
Wall time: 633 ms


(458913, 13, 177)

In [22]:
mask = np.arange(rnn_feas.shape[0])%4==0

In [23]:
tr_rnn = rnn_feas[~mask]
va_rnn = rnn_feas[mask]
tr_rnn.shape, va_rnn.shape

((344184, 13, 177), (114729, 13, 177))

For simplicity, we only use the last profile generated as new features

In [24]:
tr_rnn = tr_rnn[:,-1,:]
va_rnn = va_rnn[:,-1,:]
tr_rnn.shape, va_rnn.shape

((344184, 177), (114729, 177))

In [25]:
tr_rnn_df = cudf.DataFrame(tr_rnn,columns=[f'rnn_{i}' for i in range(tr_rnn.shape[1])])
va_rnn_df = cudf.DataFrame(va_rnn,columns=[f'rnn_{i}' for i in range(tr_rnn.shape[1])])
tr_rnn_df.shape, va_rnn_df.shape

((344184, 177), (114729, 177))

In [26]:
X_train = cudf.concat([X_train,tr_rnn_df],axis=1)
X_test = cudf.concat([X_test,va_rnn_df],axis=1)

In [27]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((344184, 354), (344184,), (114729, 354), (114729,))

In [28]:
model = get_xgb_model()
model.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)],
        verbose=100
    )
model.best_score

[0]	validation_0-logloss:0.43122	validation_0-amex_metric_np:0.70869
[58]	validation_0-logloss:0.22784	validation_0-amex_metric_np:0.77628


0.776869

### We got 0.002 improvement by adding the future profile features! That's significant improvements for default detection! It  could move the rank up by hundreds of places in the [competition](https://www.kaggle.com/competitions/amex-default-prediction/leaderboard)!!

# Save the model and write config.pbtxt for triton inference

In [38]:
from pathlib import Path

# === 1. Define model directory and variables ===
model_dir = 'amex_xgb'
Path(f'{model_dir}/1').mkdir(parents=True, exist_ok=True)

# Get variables from the test set
features = X_test.shape[1]
num_classes = y_test.unique().shape[0]
MAX_MEMORY_BYTES = 60_000_000
bytes_per_sample = (features + num_classes) * 4
max_batch_size = MAX_MEMORY_BYTES // bytes_per_sample

# === 2. Define and write config.pbtxt ===
def generate_config(model_dir, max_batch_size, features, num_classes, deployment_type='gpu'):
    if deployment_type.lower() == 'cpu':
        instance_kind = 'KIND_CPU'
    else:
        instance_kind = 'KIND_GPU'

    config_text = f"""backend: "python"
name: "amex_xgb"
max_batch_size: {max_batch_size}
input [
 {{
   name: "input__0"
   data_type: TYPE_FP32
   dims: [ {features} ]
 }}
]
output [
 {{
   name: "output__0"
   data_type: TYPE_FP32
   dims: [ {num_classes} ]
 }}
]
instance_group [{{ kind: {instance_kind} }}]
"""
    
    config_path = os.path.join(model_dir, 'config.pbtxt')
    with open(config_path, 'w') as file_:
        file_.write(config_text)
    print(f"config.pbtxt written to {config_path}")

# Call the function to write the file
generate_config(model_dir, max_batch_size, features, num_classes)

# === 3. Define and write model.py ===
model_py_code = """
import triton_python_backend_utils as pb_utils
import xgboost as xgb
import numpy as np
import json
import os 

class TritonPythonModel:
    def initialize(self, args):
        self.model = xgb.XGBClassifier()

        script_dir = os.path.dirname(__file__)
        model_path = os.path.join(script_dir, 'xgboost.model') 
        self.model.load_model(model_path)
        
        # Check config to see if we should use GPU
        model_config = json.loads(args['model_config'])
        instance_group = model_config.get('instance_group', [{}])[0]
        if instance_group.get('kind') == 'KIND_GPU':
            self.model.set_params(device='cuda')

    def execute(self, requests):
        responses = []
        for request in requests:
            input_tensor = pb_utils.get_input_tensor_by_name(request, "input__0")
            input_data = input_tensor.as_numpy()
            predictions = self.model.predict_proba(input_data)
            output_tensor = pb_utils.Tensor("output__0", predictions.astype(np.float32))
            response = pb_utils.InferenceResponse(output_tensors=[output_tensor])
            responses.append(response)
        return responses

    def finalize(self):
        pass
"""

with open(f"{model_dir}/1/model.py", "w") as f:
    f.write(model_py_code)
print(f"model.py script written to {model_dir}/1/model.py")

# === 4. Define and write requirements.txt ===
with open(f"{model_dir}/1/requirements.txt", "w") as f:
    f.write("xgboost\n")
print(f"requirements.txt written to {model_dir}/1/requirements.txt")

# === 5. Save the actual model file ===
# (This uses the 'model' variable from your training cell)
model.save_model(f'{model_dir}/1/xgboost.model')
print(f"xgboost.model saved to {model_dir}/1/xgboost.model")

config.pbtxt written to amex_xgb/config.pbtxt
model.py script written to amex_xgb/1/model.py
requirements.txt written to amex_xgb/1/requirements.txt
xgboost.model saved to amex_xgb/1/xgboost.model


/home/drollins/miniconda3/envs/rapids-modern/lib/python3.13/site-packages/xgboost/sklearn.py:1028: UserWarning: [12:28:11] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1758751550967/work/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)


In [39]:
!mkdir -p model_repository
!rm -rf model_repository/amex_xgb
!mv amex_xgb model_repository/
!mv -n AutoRegressiveRNN model_repository/
!find model_repository -type d -name ".ipynb_checkpoints" -exec rm -r {} +

print("Model repository is clean and updated.")

Model repository is clean and updated.
